# Jev query understanding with BM25 category boosting

This notebook corresponds to `configs/ecom_class/ecom_choice_single_jev.yml` and builds an ecommerce search strategy in two stages:

1. Jev classifies each query into a product category using a structured Choice question.
2. BM25 searches the WANDS catalog and adds a category boost to matching documents.

The final strategy mirrors `configs/ecom_class/ecom_choice_single_jev.yml`: title and description BM25 scores are combined, then documents matching Jev's category receive a fixed `+10` boost. `Unknown` means no category boost is applied.

The classification and search code is written below rather than imported from this repository so the notebook can teach the complete flow in a fresh Colab runtime.

## Install the notebook dependencies

`cheat-at-search` provides the WANDS data and evaluation helpers. Jev is accessed through TypeSafe's Python SDK. Pinning the Git dependency keeps the dataset and helper APIs reproducible.

In [ ]:
!pip install -q git+https://github.com/softwaredoug/cheat-at-search.git@2cfcbf60f54f07b98285b4139f5b138de2f9d774
!pip install -q 'typesafe-sdk>=0.7.1'

## Mount the shared search data directory

In Colab, the first option stores the indexed dataset on Google Drive. When Drive is unavailable, the fallback uses a local cache.

In [ ]:
from pathlib import Path

from cheat_at_search import data_dir
from cheat_at_search.data_dir import mount

try:
    mount(use_gdrive=True)
except ImportError:
    manual_path = str(Path.home() / '.search-experiments' / 'cheat-at-search')
    mount(use_gdrive=False, manual_path=manual_path)

data_dir.DATA_PATH

## Load the TypeSafe key and WANDS data

`key_for_provider('typesafe')` looks for `TYPESAFE_API_KEY` and falls back to cheat-at-search's normal key-mounting behavior. The key is passed explicitly to `TypeSafeClient`.

The WANDS corpus contains product text and category labels. Judgments tell us which products are relevant for each query, so we can evaluate the finished search strategy.

In [ ]:
import numpy as np
import pandas as pd

from cheat_at_search.data_dir import key_for_provider
from cheat_at_search.wands_data import corpus, judgments
from typesafe_sdk import Choice, TypeSafeClient

TYPESAFE_KEY = key_for_provider('typesafe')
corpus = corpus.reset_index(drop=True)
judgments = judgments.reset_index(drop=True)

corpus[['doc_id', 'title', 'description', 'category']].head(3)

In [ ]:
judgments[['query', 'doc_id', 'grade']].head(3)

## Define Jev's category criteria

A Choice is not just a list of labels. Each criterion explains when that category should win. The `Unknown` option is an explicit escape hatch for queries that do not fit these categories.

In [ ]:
CATEGORY_CRITERIA = {
    'Furniture': 'Products used to make a room suitable for living or working, such as chairs, tables, and beds.',
    'Home Improvement': 'Products and services that improve the functionality, aesthetics, or value of a home, such as tools, paint, and renovation services.',
    'Décor & Pillows': 'Products that enhance the aesthetic appeal of a space, including decorative items, pillows, and other accessories.',
    'Outdoor': 'Products designed for use outside, such as patio furniture, gardening tools, and outdoor lighting.',
    'Storage & Organization': 'Products that help organize and store items, including shelves, bins, and closet organizers.',
    'Lighting': 'Products that provide illumination, including lamps, light fixtures, and bulbs.',
    'Rugs': 'Products used to cover and decorate floors, including area rugs, runners, and mats.',
    'Bed & Bath': 'Products related to bedrooms and bathrooms, including bedding, towels, and bathroom accessories.',
    'Kitchen & Tabletop': 'Products related to kitchens and dining, including cookware, utensils, and tableware.',
    'Baby & Kids': 'Products designed for infants and children, including toys, clothing, and nursery furniture.',
    'School Furniture and Supplies': 'Products used in educational settings, including desks, chairs, and school supplies.',
    'Appliances': 'Electrical or mechanical machines designed to perform household tasks, such as refrigerators, washing machines, and microwaves.',
    'Holiday Décor': 'Seasonal decorations and accessories used to celebrate holidays.',
    'Unknown': 'No classification applies.'
}

list(CATEGORY_CRITERIA.items())[:3]

## Classify one query with Jev

The query is sent as `state`. The question itself contains the instruction and structured criteria. Jev returns the winning choice, a probability distribution, and confidence. We keep the full response here so the confidence can be inspected while the search strategy later uses only the selected category.

In [ ]:
jev = TypeSafeClient(api_key=TYPESAFE_KEY, model='jev-latest')

def classify_query(query, client=jev):
    response = client.system_one(
        state=query,
        questions={
            'category': Choice(
                instructions='Which category best describes the ecommerce search query?',
                criteria=CATEGORY_CRITERIA,
            )
        },
    )
    answer = response.choices['category']
    return {
        'query': query,
        'category': answer.choice,
        'confidence': answer.confidence,
        'probabilities': answer.probabilities,
    }

example_query = judgments.iloc[0]['query']
classify_query(example_query)

## Inspect a small batch of classifications

Every call to `classify_query` uses the TypeSafe API. Keep this limit small while experimenting, then increase it when you are ready to benchmark.

In [ ]:
CLASSIFICATION_QUERY_LIMIT = 8
sample_queries = judgments['query'].drop_duplicates().head(CLASSIFICATION_QUERY_LIMIT).tolist()
classification_df = pd.DataFrame([classify_query(query) for query in sample_queries])
classification_df[['query', 'category', 'confidence']]

## Build the SearchArray indexes

The search strategy scores title and description independently with the snowball tokenizer. It also indexes the category field so it can identify which documents receive the category boost.

In [ ]:
from searcharray import SearchArray
from cheat_at_search.tokenizers import snowball_tokenizer

for field in ['title', 'description', 'category']:
    index_name = f'{field}_snowball'
    if index_name not in corpus.columns:
        corpus[index_name] = SearchArray.index(
            corpus[field].fillna('').astype(str),
            snowball_tokenizer,
        )

corpus[['title_snowball', 'description_snowball', 'category_snowball']].head(1)

## Implement the query-understanding SearchStrategy

This is the important integration point. `search()` first computes normal BM25 relevance. It then calls Jev, finds documents whose category matches the returned choice, and adds `boost_matches` to those documents. An `Unknown` classification becomes an empty list, so it leaves the baseline BM25 scores unchanged.

In [ ]:
import hashlib
import json

from cheat_at_search.strategy import SearchStrategy
from searcharray.similarity import bm25_similarity

class JevBM25BoostedStrategy(SearchStrategy):
    def __init__(self, corpus, client, fields, field_weights, boost_matches=10, top_k=10, workers=1):
        super().__init__(corpus, top_k=top_k, workers=workers)
        self.index = corpus
        self.client = client
        self.fields = field_weights
        self.boost_matches = boost_matches
        self.category_field = 'category'
        self._classification_cache = {}
        self.similarity = bm25_similarity(k1=1.2, b=0.75)

    def enrich(self, query):
        if query not in self._classification_cache:
            answer = classify_query(query, client=self.client)
            category = answer['category']
            self._classification_cache[query] = [] if category == 'Unknown' else [category]
        return list(self._classification_cache[query])

    def _category_matches(self, category):
        terms = snowball_tokenizer(category)
        if not terms:
            return np.zeros(len(self.index), dtype=bool)
        return self.index['category_snowball'].array.score(terms) > 0

    def search(self, query, k=10):
        scores = np.zeros(len(self.index), dtype=float)
        for term in snowball_tokenizer(query):
            for field, weight in self.fields.items():
                scores += (
                    self.index[f'{field}_snowball'].array.score(
                        term, similarity=self.similarity
                    ) * weight
                )

        for category in self.enrich(query):
            scores[self._category_matches(category)] += self.boost_matches

        top_indices = np.argsort(-scores)[:k]
        return top_indices, scores[top_indices]

    @property
    def cache_key(self):
        payload = {
            'type': 'jev_bm25_boosted',
            'fields': self.fields,
            'boost_matches': self.boost_matches,
            'categories': CATEGORY_CRITERIA,
        }
        return hashlib.md5(json.dumps(payload, sort_keys=True).encode()).hexdigest()

strategy = JevBM25BoostedStrategy(
    corpus=corpus,
    client=jev,
    fields={'title': 9.4, 'description': 4.0},
    boost_matches=10,
)

strategy.search(example_query, k=3)

## Compare baseline BM25 and category-boosted results

The category boost is deliberately simple: it does not replace lexical relevance. It gives category-matching documents a fixed score advantage, so a relevant category match can move upward without forcing every result to belong to that category.

In [ ]:
result_indices, result_scores = strategy.search(example_query, k=5)
results_preview = corpus.iloc[result_indices][['doc_id', 'title', 'category']].copy()
results_preview['score'] = result_scores
results_preview

## Evaluate the finished search strategy

`run_strategy` calls the same `search()` method used in the single-query example, but compares its ranked results to the WANDS judgments. Start with a small query count while developing. Set `NUM_QUERIES = None` to run the complete benchmark; that will make one Jev classification request per distinct query not already cached.

In [ ]:
from cheat_at_search.search import run_strategy

NUM_QUERIES = 8  # use None for the full WANDS judgment set
graded = run_strategy(
    strategy,
    judgments,
    num_queries=NUM_QUERIES,
    seed=42,
    show_progress=True,
    cache=False,
)

graded.head()

## Summarize NDCG and MRR

NDCG rewards highly relevant documents appearing near the top of the list. MRR measures how early the first relevant result appears. These metrics let us evaluate the complete Jev-plus-BM25 pipeline rather than classification accuracy in isolation.

In [ ]:
from cheat_at_search.search import mrrs, ndcgs

ndcg_series = ndcgs(graded)
mrr_series = mrrs(graded)

pd.DataFrame({
    'metric': ['NDCG@10', 'MRR'],
    'value': [ndcg_series.mean(), mrr_series.mean()],
})